## Synthetic data for ASQA

Since HotpotQA provides short-form answers, we apply an answer verbalization step to transform entity-based answers into complete natural language sentences, ensuring compatibility with LLM-based evaluation metrics such as RAGAs.

In [1]:
import pandas as pd
import numpy as np
import os
import dotenv
import time
import ast
from google import genai
import google.genai
from google.genai import types
from openai import OpenAI
import asyncio
import ipywidgets
from asyncio import Semaphore
from tqdm.auto import tqdm  # Auto-select best version (notebook or console)
from kaggle_secrets import UserSecretsClient

In [2]:
# load dataset
asqa = pd.read_csv("/kaggle/input/datasets/vnphngqunh/asqa-dataset/train.csv", encoding='utf-8-sig')

In [3]:
asqa.head()

,id,question,answer,context,supporting_facts
0,asqa_0,When does the new bunk'd come out?,The new bunk'd episode 41 comes out on April 2...,"{'title': [""List of Bunk'd episodes"", 'QA_1', ...","{'title': [""List of Bunk'd episodes""]}"
1,asqa_1,Who won the 2016 ncaa football national champi...,The 2015 - 2016 season's ncaa national footbal...,{'title': ['2015 College Football Playoff Nati...,{'title': ['2016 College Football Playoff Nati...
2,asqa_2,When was the last time the death penalty was u...,The last time the death penalty was used in pa...,{'title': ['Capital punishment in Pennsylvania...,{'title': ['Gary M. Heidnik']}
3,asqa_3,Where will failure of the left ventricle cause...,"""Backward"" failure of the left ventricle cause...","{'title': ['Heart failure', 'QA_1', 'QA_2'], '...",{'title': ['Heart failure']}
4,asqa_4,Who won the war between ethiopia and italy?,The first war between Italy and Ethiopia took ...,"{'title': ['Second Italo-Ethiopian War', 'Firs...","{'title': ['First Italo-Ethiopian War', 'Secon..."


In [4]:
asqa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4353 entries, 0 to 4352
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                4353 non-null   object
 1   question          4353 non-null   object
 2   answer            4353 non-null   object
 3   context           4353 non-null   object
 4   supporting_facts  4353 non-null   object
dtypes: object(5)
memory usage: 170.2+ KB


In [5]:
print(asqa.loc[[1]])


       id                                           question  \
1  asqa_1  Who won the 2016 ncaa football national champi...   

                                              answer  \
1  The 2015 - 2016 season's ncaa national footbal...   

                                             context  \
1  {'title': ['2015 College Football Playoff Nati...   

                                    supporting_facts  
1  {'title': ['2016 College Football Playoff Nati...  


In [6]:
user_secrets = UserSecretsClient()
GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")
OPENAI_API_KEY = user_secrets.get_secret("OPENAI_API_KEY")

# client = genai.Client(api_key=GOOGLE_API_KEY)
client = OpenAI(api_key=OPENAI_API_KEY)
SLEEP_TIME = 1


In [7]:
for m in client.models.list():
    print(m.id)

text-embedding-ada-002
whisper-1
gpt-3.5-turbo
tts-1
gpt-3.5-turbo-16k
gpt-4-0613
gpt-4
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
dall-e-3
dall-e-2
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
gpt-4o-audio-preview
gpt-4o-realtime-preview
omni-moderation-latest
omni-moderation-2024-09-26
gpt-4o-realtime-preview-2024-12-17
gpt-4o-audio-preview-2024-12-17
gpt-4o-mini-realtime-preview-2024-12-17
gpt-4o-mini-audio-preview-2024-12-17
o1-2024-12-17
o1
gpt-4o-mini-realtime-preview
gpt-4o-mini-audio-preview
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
o1-pro-2025-03-19
o1-pro
gpt-4o-mini-tts
o3-2025-04-16
o4-mini-2025-04-16
o3
o4-mini
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2

In [8]:
def parse_supporting_facts(supporting_facts_raw, context_raw):
    try:
        # ===== clean string =====
        sf_clean = supporting_facts_raw.replace('""', '"')
        ctx_clean = context_raw.replace('""', '"')

        sf = ast.literal_eval(sf_clean)
        ctx = ast.literal_eval(ctx_clean)

        # ===== lấy danh sách title cần dùng =====
        sf_titles = set(sf.get("title", []))

        ctx_titles = ctx.get("title", [])
        ctx_sentences = ctx.get("sentences", [])

        facts = []

        # ===== lọc đúng doc theo title =====
        for title, sents in zip(ctx_titles, ctx_sentences):
            if title in sf_titles or str(title).startswith("QA"):
                joined_sents = " ".join(sents)
                
                facts.append(f"- ({title}) {joined_sents}")

        return "\n".join(facts)

    except Exception as e:
        return ""

In [9]:
PROMPT_TEMPLATE = """
You are given a question, a correct answer, and supporting facts extracted from reliable sources.

Your task is to generate a hallucinated answer.

=====================
Requirements:
- The answer MUST be factually incorrect based on the supporting facts
- The answer MUST directly conflict with at least one key fact (e.g., date, number, entity)
- The answer should remain fluent, natural, and confident
- Keep the style and length similar to the original answer (1–2 sentences)
- DO NOT mention uncertainty or say it is incorrect
- DO NOT use any special formatting (e.g., bold, italics, bullet points, markdown, or symbols)
- Output plain text only

=====================
Hallucination Strategy:
- Modify ONLY 1–2 key facts from the supporting facts
- Keep all other details consistent with the original answer
- Prefer subtle contradictions over obvious errors

=====================
Input:

Question:
{question}

Correct Answer:
{correct_answer}

Supporting Facts:
{supporting_facts}

=====================
Output:
Return ONLY the hallucinated answer.
"""

PROMPT_SYSTEM = """You are an AI that generates hallucinated answers for evaluation purposes.

Follow all instructions strictly:
- The answer MUST be factually incorrect based on the supporting facts
- The answer MUST directly conflict with at least one key fact (e.g., date, number, entity)
- The answer should remain fluent, natural, and confident
- Keep the style and length similar to the original answer (1–2 sentences)
- DO NOT mention uncertainty or say it is incorrect
- DO NOT use any special formatting
- Output plain text only

Hallucination Strategy:
- Modify ONLY 1–2 key facts
- Keep all other details consistent
- Prefer subtle contradictions over obvious errors
"""

PROMPT_USER = """Question:
{question}

Correct Answer:
{correct_answer}

Supporting Facts:
{supporting_facts}

Return ONLY the hallucinated answer.
"""

In [10]:
def extract_text(response):
    try:
        return "".join(
            part.text
            for cand in response.candidates
            for part in cand.content.parts
            if hasattr(part, "text") and part.text
        ).strip()
    except:
        return ""

In [11]:
def synthetic(row, client, model="gpt-4o-mini"):
    facts_text = parse_supporting_facts(
        row["supporting_facts"],
        row["context"]
    )
    
    prompt_user = PROMPT_USER.format(
        question=row["question"].strip(),
        correct_answer=row["answer"].strip(),
        supporting_facts=facts_text
    )

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": PROMPT_SYSTEM},
                {"role": "user", "content": prompt_user}
            ],
            temperature=0.7
        )
        text = response.choices[0].message.content
        # basic validation
        if not text or text == row["answer"]:
           return None
        
        return text
    
    except Exception:
        return None



In [12]:
row = asqa.iloc[3]  # lấy sample đầu tiên

hallu = synthetic(
    row=row,
    client=client,
    model="gpt-4o-mini"
)

print("=== QUESTION ===")
print(row["question"])

print("\n=== ORIGINAL ANSWER ===")
print(row["answer"])

print("\n=== HALLUCINATED ANSWER ===")
print(hallu)

=== QUESTION ===
Where will failure of the left ventricle cause increased pressure?

=== ORIGINAL ANSWER ===
"Backward" failure of the left ventricle causes congestion of the lungs' blood vessels, and therefore causes increased pressure in the lungs. These symptoms are predominantly respiratory in nature.

=== HALLUCINATED ANSWER ===
"Backward" failure of the left ventricle causes congestion of the liver's blood vessels, and therefore causes increased pressure in the liver. These symptoms are predominantly digestive in nature.


In [13]:
asqa = asqa.reset_index(drop=True)

In [14]:
def synthetic_batch(df, client, model="gpt-4o-mini",
                    checkpoint_path=None, checkpoint_freq=100):
    df = df.copy()

    hallu_answers = []
    ids = []

    # ===== Resume =====
    start_idx = 0
    if checkpoint_path and os.path.exists(checkpoint_path):
        checkpoint_df = pd.read_csv(checkpoint_path)
        hallu_answers = checkpoint_df["answer"].tolist()
        new_ids = checkpoint_df["id"].tolist()
        start_idx = len(hallu_answers)
        print(f"Resuming from index {start_idx}")

    # ===== Progress bar =====
    pbar = tqdm(
        total=len(df),
        initial=start_idx,
        desc="Generating hallucinations",
        unit="sample"
    )

    for idx, row in df.iterrows():
        if idx < start_idx:
            continue
        
        try:
            hallucinated = synthetic(row=row, client=client, model=model)
            if not hallucinated:
                hallucinated = ""
            new_id = str(row.id)  

        except Exception as e:
            print(f"\nError at index {idx}: {e}")

            if checkpoint_path:
                temp_df = pd.DataFrame({
                    "id": str(row.id),
                    "answer": hallu_answers
                })
                temp_df.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")
                print(f"Checkpoint saved at index {idx}")

            break

        hallu_answers.append(hallucinated)
        ids.append(new_id)

        pbar.update(1)
        time.sleep(SLEEP_TIME)

        # ===== checkpoint =====
        if (checkpoint_path and (idx + 1) % checkpoint_freq == 0) or idx==(len(df)-1):
            temp_df = pd.DataFrame({
                "id": ids,
                "answer": hallu_answers
            })
            temp_df.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")
            print(f"💾 checkpoint saved at {idx}")

    pbar.close()

    return df

In [15]:
def build_hallu_dataset(asqa, client, model="gpt-4o-mini", checkpoint_path=None, hallu_path=None, save_path=None):
    asqa = asqa.copy()

    # ==============================
    # 1. LOOP BATCH TỚI KHI ĐỦ CHECKPOINT
    # ==============================
    while True:
        print("\n🚀 Running batch...")

        df_hallu = synthetic_batch(
            asqa,
            client,
            checkpoint_path=checkpoint_path,
            checkpoint_freq=50
        )

        if not os.path.exists(checkpoint_path):
            continue

        ckpt = pd.read_csv(checkpoint_path)

        # check id cuối
        if len(ckpt) == len(asqa):
            print("✅ Checkpoint đủ length")
            break
        else:
            print(f"⏳ checkpoint {len(ckpt)}/{len(asqa)} → chạy tiếp")

    # ==============================
    # 2. LOAD CHECKPOINT
    # ==============================
    df_hallu = pd.read_csv(checkpoint_path)

    # đảm bảo align id
    df_hallu["id"] = asqa["id"].values

    # ==============================
    # 3. FILTER OUTPUT RÁC
    # ==============================
    df_hallu = df_hallu[
        df_hallu["answer"].notnull() &
        (df_hallu["answer"].str.len() >= 5)
    ].copy()

    # ==============================
    # 4. RETRY NHỮNG ID THIẾU
    # ==============================
    existing_ids = set(df_hallu["id"])
    missing_ids = [i for i in asqa["id"] if i not in existing_ids]

    print(f"🔁 Missing: {len(missing_ids)}")

    if missing_ids:
        df_missing = asqa[asqa["id"].isin(missing_ids)]

        df_retry = synthetic_batch(df_missing, client)

        df_retry = df_retry[
            df_retry["answer"].notnull() &
            (df_retry["answer"].str.len() >= 5)
        ]

        df_retry["id"] = df_missing["id"].values

        df_hallu = pd.concat([df_hallu, df_retry], ignore_index=True)

    temp = asqa.copy()
    temp["answer"] = df_hallu["answer"]
    df_hallu = temp
    
    df_hallu.to_csv(hallu_path, index=False, encoding="utf-8-sig")

    # ==============================
    # 5. GẮN ID + LABEL
    # ==============================
    df_hallu["id"] = df_hallu["id"] + "b"
    df_hallu["label"] = 0

    df_orig = asqa.copy()
    df_orig["label"] = 1

    # ==============================
    # 6. CONCAT
    # ==============================
    df_final = pd.concat([df_orig, df_hallu], ignore_index=True)
    df_final.to_csv(save_path, index=False, encoding="utf-8-sig")
    
    return df_final, df_hallu

In [16]:
asqa_final, asqa_hallu = build_hallu_dataset(asqa, client,  model="gpt-4o-mini",
                                            checkpoint_path='/kaggle/working/checkpoint.csv', 
                                            hallu_path='/kaggle/working/hallu_asqa.csv', 
                                            save_path='/kaggle/working/labeled_asqa.csv')


🚀 Running batch...


Generating hallucinations:   0%|          | 0/4353 [00:00<?, ?sample/s]

💾 checkpoint saved at 49
💾 checkpoint saved at 99
💾 checkpoint saved at 149
💾 checkpoint saved at 199
💾 checkpoint saved at 249
💾 checkpoint saved at 299
💾 checkpoint saved at 349
💾 checkpoint saved at 399
💾 checkpoint saved at 449
💾 checkpoint saved at 499
💾 checkpoint saved at 549
💾 checkpoint saved at 599
💾 checkpoint saved at 649
💾 checkpoint saved at 699
💾 checkpoint saved at 749
💾 checkpoint saved at 799
💾 checkpoint saved at 849
💾 checkpoint saved at 899
💾 checkpoint saved at 949
💾 checkpoint saved at 999
💾 checkpoint saved at 1049
💾 checkpoint saved at 1099
💾 checkpoint saved at 1149
💾 checkpoint saved at 1199
💾 checkpoint saved at 1249
💾 checkpoint saved at 1299
💾 checkpoint saved at 1349
💾 checkpoint saved at 1399
💾 checkpoint saved at 1449
💾 checkpoint saved at 1499
💾 checkpoint saved at 1549
💾 checkpoint saved at 1599
💾 checkpoint saved at 1649
💾 checkpoint saved at 1699
💾 checkpoint saved at 1749
💾 checkpoint saved at 1799
💾 checkpoint saved at 1849
💾 checkpoint saved at 1

In [17]:
asqa_final.head(20)

,id,question,answer,context,supporting_facts,label
0,asqa_0,When does the new bunk'd come out?,The new bunk'd episode 41 comes out on April 2...,"{'title': [""List of Bunk'd episodes"", 'QA_1', ...","{'title': [""List of Bunk'd episodes""]}",1
1,asqa_1,Who won the 2016 ncaa football national champi...,The 2015 - 2016 season's ncaa national footbal...,{'title': ['2015 College Football Playoff Nati...,{'title': ['2016 College Football Playoff Nati...,1
2,asqa_2,When was the last time the death penalty was u...,The last time the death penalty was used in pa...,{'title': ['Capital punishment in Pennsylvania...,{'title': ['Gary M. Heidnik']},1
3,asqa_3,Where will failure of the left ventricle cause...,"""Backward"" failure of the left ventricle cause...","{'title': ['Heart failure', 'QA_1', 'QA_2'], '...",{'title': ['Heart failure']},1
4,asqa_4,Who won the war between ethiopia and italy?,The first war between Italy and Ethiopia took ...,"{'title': ['Second Italo-Ethiopian War', 'Firs...","{'title': ['First Italo-Ethiopian War', 'Secon...",1
5,asqa_5,Who played bonnie in gone with the wind?,The 1939 film gone with the wind's character B...,"{'title': ['Gone with the Wind (film)', 'Gone ...","{'title': ['Cammie King', 'Gone with the Wind ...",1
6,asqa_6,Premier league record for most wins in a row?,"The English Premier league, a number of teams ...",{'title': ['Premier League records and statist...,{'title': ['Premier League records and statist...,1
7,asqa_7,What episode does goku become super saiyan 3?,"In the Dragon Ball Z Kai anime series, goku be...",{'title': ['List of Dragon Ball Z Kai episodes...,{'title': ['List of Dragon Ball Z Kai episodes']},1
8,asqa_8,Who published harry potter and the prisoner of...,"The third book in the ""Harry Potter"" series Ha...",{'title': ['Harry Potter and the Prisoner of A...,{'title': ['Harry Potter and the Prisoner of A...,1
9,asqa_9,When does doctor strange get the infinity stone?,"In the animated direct-to-video film, Doctor S...","{'title': ['Infinity Gems', 'Eye of Agamotto',...","{'title': ['Doctor Strange', 'Infinity Gems', ...",1
